<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/x-segmentation-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the course · Segmentation 2 — From pixels to separate cells

In Segmentation 1 your U-Net learned to say, for every pixel, "nucleus or not".
That is **semantic segmentation**. Project II needs more: each cell
separately, so you can count them and outline each one. That is **instance
segmentation**. This notebook turns your U-Net's mask into separate nuclei,
scores a count properly, and solves the tile-edge problem from Day 5's
starter on a big image.

This is an extension pathway: you work through it alone or with a partner,
at your own pace. You need Segmentation 1 behind you. Allow about two
hours.

How it works, as in class: 🔮 predict before you run, ✏️ replace the `...`,
✅ run the check under it. A check that fails tells you what to look at. Each
section ends with a **You're done when** line.

**GPU or not?** Colab gives each notebook a fresh computer, so the model from
Segmentation 1 is gone. Run the three cells at the top in order: set up, then
the ⬇️ download cell (the images again, 81 MB; if it stops with an error, run
it again), then the cell that trains the same U-Net again: a few minutes on
the free CPU or on a T4 GPU (which does more steps). Read section 1 while it
trains.

In [ ]:
#@title Run this first: set up the tools { display-mode: "form" }
import os
import hashlib
import shutil
import glob
import time
import pathlib
import urllib.request
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
import tifffile
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
import torch
from torch import nn
import torch.nn.functional as F


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = device.type == "cuda"
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
torch.manual_seed(0)
np.random.seed(0)

DATA = pathlib.Path("data/bbbc039")

⬇️ **Download the data (81 MB).** Only run the next cell if you are going to use this data. It downloads 81 MB (the BBBC039 images and masks), so skip it if you are only reading. It is safe to run twice. "Run all" runs it too, so use "Run all" only when you mean to do the whole notebook.

In [ ]:
# ⬇️ Download cell: BBBC039 images and masks (81 MB)
BBBC_SHA256 = {
    "images": "6f30a5d4fe38c928ded972704f085975f8dc0d65d9aa366df00e5a9d449fddd7",
    "masks": "f9e6043d8ca56344a4886f96a700d804d6ee982f31e2b2cd3194af2a053c2710",
}


def fetch(name):
    """Download and unzip one BBBC039 file, once."""
    DATA.mkdir(parents=True, exist_ok=True)
    zpath = DATA / f"{name}.zip"
    if not fetch_from_mirror(f"https://data.tk.sg/bbbc039/{name}.zip", zpath, BBBC_SHA256[name]):
        urllib.request.urlretrieve(f"https://data.broadinstitute.org/bbbc/BBBC039/{name}.zip", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA)
    zpath.unlink()


if (DATA / "images").exists() and (DATA / "masks").exists():
    print("BBBC039 is already downloaded. Nothing to do.")
else:
    for part in ("images", "masks"):
        if not (DATA / part).exists():
            fetch(part)
    print("Done: BBBC039 downloaded (81 MB).")

In [ ]:
#@title Then run this: load the data and train the U-Net from Segmentation 1 (about 5 minutes) { display-mode: "form" }
require_data("BBBC039", DATA / "images", DATA / "masks")
image_files = sorted(glob.glob(str(DATA / "images" / "*.tif")))


def load_image(path):
    img = tifffile.imread(path).astype(np.float32)
    lo, hi = np.percentile(img, [1, 99.5])
    return np.clip((img - lo) / (hi - lo), 0, 1).astype(np.float32)


def load_labels(path):
    colours = iio.imread(path.replace("images", "masks").replace(".tif", ".png"))[:, :, 0]
    labels, total = np.zeros(colours.shape, dtype=int), 0
    for colour in (1, 2, 3):
        blobs, n = ndimage.label(colours == colour)
        labels[blobs > 0] = blobs[blobs > 0] + total
        total += n
    return labels


images = [load_image(f) for f in image_files]
label_images = [load_labels(f) for f in image_files]
train_imgs, train_labs = images[:150], label_images[:150]
val_imgs, val_labs = images[150:], label_images[150:]
if QUICK:
    val_imgs, val_labs = val_imgs[:4], val_labs[:4]


def block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
    )


class UNet(nn.Module):
    def __init__(self, c=16):
        super().__init__()
        self.down1, self.down2, self.down3 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c)
        self.up2, self.join2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), block(4 * c, 2 * c)
        self.up1, self.join1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), block(2 * c, c)
        self.head = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(F.max_pool2d(x1, 2))
        x3 = self.down3(F.max_pool2d(x2, 2))
        y = self.join2(torch.cat([self.up2(x3), x2], dim=1))
        y = self.join1(torch.cat([self.up1(y), x1], dim=1))
        return self.head(y)


def random_batch(n, size):
    xs, ys = [], []
    for k in np.random.randint(len(train_imgs), size=n):
        r = np.random.randint(train_imgs[k].shape[0] - size + 1)
        c = np.random.randint(train_imgs[k].shape[1] - size + 1)
        xs.append(train_imgs[k][r:r + size, c:c + size])
        ys.append(train_labs[k][r:r + size, c:c + size] > 0)
    return (torch.tensor(np.array(xs))[:, None].to(device),
            torch.tensor(np.array(ys), dtype=torch.float32)[:, None].to(device))


STEPS = 5 if QUICK else (1500 if GPU else 250)
BATCH, SIZE = (32, 256) if GPU else (8, 128)
model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
start = time.time()
model.train()
for step in range(STEPS):
    x, y = random_batch(BATCH, SIZE)
    optimizer.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(x), y)
    loss.backward()
    optimizer.step()
    if step % 50 == 0 or step == STEPS - 1:
        print(f"step {step:4d}  loss {loss.item():.3f}  {time.time() - start:.0f} s")
model.eval()


def probability(img):
    """The U-Net's probability of 'nucleus' at every pixel of a whole image."""
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(img)[None, None].to(device)))[0, 0].cpu().numpy()


val_probs = [probability(img) for img in val_imgs]
val_masks = [p > 0.5 for p in val_probs]
print("ready:", len(val_imgs), "validation images, each with the U-Net's mask")

## 1. Counting blobs undercounts

The quickest way to count nuclei in a mask is Day 2's: `ndimage.label`
numbers each separate blob, and the count is the largest label. Here it is on
one validation image, next to the true count from the hand-drawn labels.

In [ ]:
blobs, n_blobs = ndimage.label(val_masks[0])
print("blobs in the U-Net's mask:", n_blobs, "   nuclei drawn by hand:", val_labs[0].max())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(val_imgs[0], cmap="gray"); axes[0].set_title("validation image 0")
axes[1].imshow(np.ma.masked_equal(blobs, 0), cmap="tab20", interpolation="nearest"); axes[1].set_title("blobs, one colour each")
for ax in axes:
    ax.axis("off")
plt.show()

🔮 **Predict.** Over all the validation images, will the blob count be too
high or too low, and by roughly how much? Think about what happens where two
nuclei touch.

In [ ]:
my_prediction = ""  # @param {type:"string"}

To see why, ask: how many hand-drawn nuclei sit inside one blob? Here is how
to list the labels under one region of the image:

In [ ]:
region = blobs == 1
print("hand-drawn labels under blob 1:", np.unique(val_labs[0][region]))

✏️ **Exercise 1a.** Write `nuclei_inside(true_labels, region)`: how many
different hand-drawn nuclei does a True/False `region` overlap? Leave out 0,
which is background.

In [ ]:
def nuclei_inside(true_labels, region):
    return ...

In [ ]:
demo_true = np.array([[1, 1, 0, 2, 2],
                      [1, 1, 0, 2, 2],
                      [0, 0, 0, 0, 3]])
whole = np.ones(demo_true.shape, dtype=bool)
check(is_done(nuclei_inside(demo_true, whole)), "nuclei_inside still returns `...`.")
check(nuclei_inside(demo_true, whole) == 3, "The whole demo overlaps nuclei 1, 2 and 3: the answer is 3. Did you leave out 0 (background)?")
check(nuclei_inside(demo_true, demo_true == 1) == 1, "A region exactly on nucleus 1 overlaps 1 nucleus.")
check(nuclei_inside(demo_true, demo_true == 0) == 0, "A region on background overlaps no nuclei.")

In [ ]:
merged = [nuclei_inside(val_labs[0], blobs == b) for b in range(1, n_blobs + 1)]
print(f"{sum(m > 1 for m in merged)} of {n_blobs} blobs cover two or more hand-drawn nuclei")
totals = [(ndimage.label(m)[1], lab.max()) for m, lab in zip(val_masks, val_labs)]
print("over all validation images: blobs", sum(t[0] for t in totals), "  hand-drawn nuclei", sum(t[1] for t in totals))

**You're done when** you know how many blobs in image 0 contain more than one
nucleus, and whether the blob count is too high or too low overall.

## 2. Split the blobs: distance transform and watershed

Two touching round nuclei make a blob shaped like a peanut. The trick for
splitting it:

1. The **distance transform** gives every pixel inside the mask its distance
   to the nearest background pixel. The centre of each nucleus is far from
   the edge, so it becomes a hill; the narrow waist between two nuclei is
   low.
2. Find the hilltops (**local maxima**) with `peak_local_max`. Each one is a
   **marker**: the seed of one nucleus. `min_distance` stops two hilltops
   closer than that many pixels both counting. `labels=` says which blob
   each pixel belongs to, so the hilltops are found in each blob on its
   own: give it the numbered blobs from `ndimage.label(mask)[0]`, not the
   True/False mask. (With True/False, every blob counts as one region, and
   a small nucleus next to a bigger one can lose its hilltop to the bigger
   one's.) `exclude_border=False` lets
   a nucleus touching the image's edge have a marker too (by default
   `peak_local_max` ignores hilltops near the edge, and a blob with no
   marker is dropped).
3. **Watershed** floods outward from the markers, like water rising from
   several springs, and draws a border where two floods meet. It runs on the
   *negative* distance, so the hilltops become valleys that fill first.

Here are the three steps on a made-up peanut: two overlapping discs.

In [ ]:
yy, xx = np.mgrid[0:60, 0:90]
peanut = ((yy - 30) ** 2 + (xx - 30) ** 2 < 18 ** 2) | ((yy - 30) ** 2 + (xx - 58) ** 2 < 18 ** 2)

distance = ndimage.distance_transform_edt(peanut)
peaks = peak_local_max(distance, min_distance=10, labels=ndimage.label(peanut)[0], exclude_border=False)
markers = np.zeros(peanut.shape, dtype=int)
for k, (r, c) in enumerate(peaks, start=1):
    markers[r, c] = k
split = watershed(-distance, markers, mask=peanut)

print("blobs before:", ndimage.label(peanut)[1], "  after watershed:", split.max())
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(peanut, cmap="gray"); axes[0].set_title("mask: one blob")
axes[1].imshow(distance, cmap="magma"); axes[1].plot(peaks[:, 1], peaks[:, 0], "c+", ms=15); axes[1].set_title("distance, with markers")
axes[2].imshow(split, cmap="tab10", interpolation="nearest"); axes[2].set_title("watershed: two nuclei")
for ax in axes:
    ax.axis("off")
plt.show()

✏️ **Exercise 2a.** Write `split_nuclei(mask, min_distance)`, the same three
steps for any True/False mask. Return the label image from `watershed`.

In [ ]:
def split_nuclei(mask, min_distance=10):
    distance = ...
    peaks = ...
    markers = ...
    return ...

In [ ]:
three = peanut.copy()
three[(yy - 30) ** 2 + (xx - 84) ** 2 < 10 ** 2] = False
three |= (yy - 10) ** 2 + (xx - 80) ** 2 < 7 ** 2
single = (yy - 30) ** 2 + (xx - 45) ** 2 < 15 ** 2
out = split_nuclei(three)
check(is_done(out), "split_nuclei still returns `...`.")
check(isinstance(out, np.ndarray) and out.shape == three.shape, "Return the label image that watershed gives, the same shape as the mask.")
check(out.max() == 3, f"The test mask has a peanut (two nuclei) and one small disc: 3 nuclei, but you found {out.max()}.")
check(split_nuclei(single).max() == 1, "A single round nucleus should stay as one.")
check(((out > 0) == three).all(), "Every mask pixel should get a label, and no background pixel. Pass mask=mask to watershed, and exclude_border=False to peak_local_max.")

Everyone continues from here (code hidden; double-click to see it after).

In [ ]:
#@title Run this to continue { display-mode: "form" }
def split_nuclei(mask, min_distance=10):
    dist = ndimage.distance_transform_edt(mask)
    seeds = np.zeros(mask.shape, dtype=int)
    for k, (r, c) in enumerate(peak_local_max(dist, min_distance=min_distance, labels=ndimage.label(mask)[0], exclude_border=False)):
        seeds[r, c] = k + 1
    return watershed(-dist, seeds, mask=mask)

🐛 **Break it.** Change `min_distance` below to 2, then to 30, and run each.
What goes wrong at each end? Which kinds of nuclei suffer?

In [ ]:
min_distance = 10
pieces = split_nuclei(val_masks[0], min_distance)
print(f"min_distance {min_distance}: {pieces.max()} nuclei   (hand-drawn: {val_labs[0].max()})")
plt.figure(figsize=(6, 4.5))
plt.imshow(np.ma.masked_equal(pieces, 0), cmap="tab20", interpolation="nearest")
plt.axis("off")
plt.show()

**You're done when** your `split_nuclei` check passes and you can name what
goes wrong with a too-small and a too-large `min_distance`.

## 3. Score a count properly

"My model found 98 nuclei and the answer is 100" sounds great, but the model
could have found 98 wrong things. A fair score checks each nucleus.

The standard rule: a predicted nucleus **matches** a hand-drawn one if their
IoU is at least 0.5. Then

- a **true positive** (TP) is a predicted nucleus that matches;
- a **false positive** (FP) is a predicted nucleus that matches nothing;
- a **false negative** (FN) is a hand-drawn nucleus nothing matched.

From those, **F1** = 2·TP / (2·TP + FP + FN): 1 is perfect, 0 is useless.
(With IoU of at least 0.5, one predicted nucleus can match at most one hand-drawn
one, so nothing is counted twice.)

Here is the IoU of one predicted nucleus with the hand-drawn nucleus it
overlaps most. This is the worked example for the exercise:

In [ ]:
pred = split_nuclei(val_masks[0])
p = 1
mine = pred == p
under = val_labs[0][mine]
under = under[under > 0]
if len(under) == 0:
    print("predicted nucleus", p, "overlaps no hand-drawn nucleus")
else:
    best = np.bincount(under).argmax()
    theirs = val_labs[0] == best
    print(f"predicted nucleus {p} overlaps hand-drawn nucleus {best} most, IoU {(mine & theirs).sum() / (mine | theirs).sum():.2f}")

✏️ **Exercise 3a.** Write `match_count(true_labels, pred_labels)`. Loop over
every predicted label from 1 to `pred_labels.max()`, work out its IoU with the
hand-drawn nucleus it overlaps most (as above), and count it as a TP when the
IoU is at least 0.5. Return `(tp, fp, fn)`.

In [ ]:
def match_count(true_labels, pred_labels, need=0.5):
    tp = 0
    ...  # loop over the predicted nuclei, adding 1 to tp for each match
    fp = ...
    fn = ...
    return tp, fp, fn


def f1(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0

In [ ]:
truth = np.zeros((10, 20), dtype=int)
truth[1:5, 1:5], truth[1:5, 8:12], truth[6:9, 15:18] = 1, 2, 3
guess = np.zeros_like(truth)
guess[1:5, 1:5] = 1        # exactly nucleus 1
guess[2:5, 9:12] = 2       # most of nucleus 2: IoU 9/16
guess[7:9, 0:3] = 3        # nothing there
check(is_done(*match_count(truth, guess)), "match_count still returns `...`.")
check(match_count(truth, guess) == (2, 1, 1), f"Expected (2, 1, 1): two matches, one false positive, one missed nucleus. You returned {match_count(truth, guess)}.")
check(match_count(truth, truth) == (3, 0, 0), "Labels compared with themselves: every nucleus matches.")
small = np.zeros_like(truth)
small[1:3, 1:3] = 1
check(match_count(truth, small) == (0, 1, 3), "A 2×2 guess on a 4×4 nucleus has IoU 0.25: not a match.")

Everyone continues from here. The hidden version gives the same numbers,
written to be fast on whole images.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def match_count(true_labels, pred_labels, need=0.5):
    n_true, n_pred = int(true_labels.max()), int(pred_labels.max())
    if n_true == 0 or n_pred == 0:
        return 0, n_pred, n_true
    pairs = np.bincount(true_labels.ravel() * (n_pred + 1) + pred_labels.ravel(),
                        minlength=(n_true + 1) * (n_pred + 1)).reshape(n_true + 1, n_pred + 1)
    inter = pairs[1:, 1:]
    size_t, size_p = pairs[1:, :].sum(1), pairs[:, 1:].sum(0)
    iou_table = inter / (size_t[:, None] + size_p[None, :] - inter)
    tp = int((iou_table >= need).any(axis=0).sum())
    return tp, n_pred - tp, n_true - tp

🔮 **Predict.** Blob counting against watershed splitting: which gets the
higher F1 over the validation images, and which of FP or FN changes most?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
def total_score(make_labels):
    tp = fp = fn = 0
    for mask, lab in zip(val_masks, val_labs):
        a, b, c = match_count(lab, make_labels(mask))
        tp, fp, fn = tp + a, fp + b, fn + c
    return tp, fp, fn


for name, method in (("blobs", lambda m: ndimage.label(m)[0]), ("watershed", split_nuclei)):
    tp, fp, fn = total_score(method)
    print(f"{name:10s} TP {tp:5d}  FP {fp:4d}  FN {fn:4d}   F1 {f1(tp, fp, fn):.3f}")

**You're done when** you have an F1 for both methods and can say which kind of
error watershed fixed.

## 4. Big images: overlapping tiles

Project II's images are far too big to give a model in one go, so they are cut
into tiles, as on Day 2 and Day 5, and each tile is segmented on its own.
Here is a stand-in: four validation images joined into one 1040×1392 picture.

In [ ]:
big = np.block([[val_imgs[0], val_imgs[1]], [val_imgs[2], val_imgs[3]]])
quarters, offset = [], 0
for lab in val_labs[:4]:
    quarters.append(np.where(lab > 0, lab + offset, 0))   # renumber so labels don't repeat
    offset += lab.max()
big_true = np.block([[quarters[0], quarters[1]], [quarters[2], quarters[3]]])
print("big image", big.shape, "with", len(np.unique(big_true)) - 1, "hand-drawn nuclei")

Day 5's starter found the problem: a nucleus on the line between two tiles
is cut in two, and each tile counts its half as a whole nucleus. The fix has
two parts.

1. **Overlap the tiles.** Each tile starts `step` pixels after the last, with
   `step` smaller than the tile, so every nucleus lies well inside at least
   one tile.
2. **Give every pixel one owner.** Split each overlap down the middle. A tile
   **owns** the part of the image closer to its middle than to its
   neighbour's, and keeps a nucleus only if the nucleus's centre is in the
   part it owns. Every nucleus is then kept exactly once, by a tile that saw
   all of it.

Cutting tiles side by side, as on Day 5, means each tile starts where the
last ended. These are the starting rows of 256-pixel tiles down 1,040 rows:

In [ ]:
print("side by side:", list(range(0, 1040 - 256 + 1, 256)), "+ one more at", 1040 - 256, "to reach the end")

✏️ **Exercise 4a.** Write `tile_starts(length, tile, step)`: the starting
positions `0, step, 2·step, …` that fit (start + tile ≤ length), and then, if
the last tile doesn't reach the end, one more start at `length - tile`.

In [ ]:
def tile_starts(length, tile, step):
    starts = ...
    return starts

In [ ]:
check(is_done(tile_starts(1040, 256, 192)), "tile_starts still returns `...`.")
check(tile_starts(1040, 256, 192) == [0, 192, 384, 576, 768, 784], f"For 1040 with 256-pixel tiles every 192: [0, 192, 384, 576, 768, 784], not {tile_starts(1040, 256, 192)}.")
check(tile_starts(512, 256, 256) == [0, 256], "512 splits exactly into two 256 tiles: no extra start.")
check(tile_starts(256, 256, 192) == [0], "An image the size of one tile needs one tile.")

Everyone continues from here. `owned` splits each overlap down the middle:
for tiles starting at 0 and 192, the cut is halfway through the overlap from
192 to 256, at 224. `segment_tiled` runs the U-Net and `split_nuclei` on each
tile and keeps only the nuclei whose centre the tile owns.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def tile_starts(length, tile, step):
    out = list(range(0, length - tile + 1, step))
    return out if out[-1] + tile >= length else out + [length - tile]


def owned(starts, tile, length):
    """For each tile, the (from, to) range of rows or columns it owns."""
    cuts = [0] + [(a + tile + b) // 2 for a, b in zip(starts, starts[1:])] + [length]
    return list(zip(cuts[:-1], cuts[1:]))


def segment_tiled(img, tile=256, step=192):
    """Segment a big image tile by tile; each nucleus is kept by the tile that owns its centre."""
    out, n = np.zeros(img.shape, dtype=int), 0
    rows, cols = tile_starts(img.shape[0], tile, step), tile_starts(img.shape[1], tile, step)
    for r, (r_from, r_to) in zip(rows, owned(rows, tile, img.shape[0])):
        for c, (c_from, c_to) in zip(cols, owned(cols, tile, img.shape[1])):
            lab = split_nuclei(probability(img[r:r + tile, c:c + tile]) > 0.5)
            centres = ndimage.center_of_mass(lab > 0, lab, range(1, lab.max() + 1))
            window = out[r:r + tile, c:c + tile]
            for k, (cy, cx) in enumerate(centres, start=1):
                if r_from <= r + cy < r_to and c_from <= c + cx < c_to:
                    n += 1
                    window[(lab == k) & (window == 0)] = n
    return out

🔮 **Predict.** Three ways to segment the big picture: the whole image at
once (possible here, impossible for Project II's images), tiles side by side
(`step` = 256), and overlapping tiles (`step` = 192). Which will find too
many nuclei, and why? Rank them by F1.

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
ways = {
    "whole image": split_nuclei(probability(big) > 0.5),
    "side by side": segment_tiled(big, 256, 256),
    "overlapping": segment_tiled(big, 256, 192),
}
for name, lab in ways.items():
    tp, fp, fn = match_count(big_true, lab)
    print(f"{name:13s} {lab.max():4d} nuclei   F1 {f1(tp, fp, fn):.3f}   (TP {tp}, FP {fp}, FN {fn})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name in zip(axes, ("side by side", "overlapping")):
    ax.imshow(np.ma.masked_equal(ways[name], 0) % 20, cmap="tab20", interpolation="nearest")
    for start in tile_starts(1040, 256, 256 if name == "side by side" else 192):
        ax.axhline(start, color="k", lw=0.5)
    ax.set_title(name)
    ax.axis("off")
plt.show()

**You're done when** you have the three counts and F1 scores, and have found a
nucleus cut in two in the side-by-side picture.

### Stretch: make it yours

- **Only dots.** Project II gives a centre point for each cell, not an
  outline. Make a training target with a small bright spot at each nucleus's
  centre (start from `ndimage.center_of_mass(lab > 0, lab, range(1, lab.max() + 1))`
  and `ndimage.gaussian_filter`), train the U-Net with mean squared error
  instead of cross-entropy, and count with `peak_local_max` on its output.
  Score a count by matching each found point to a true centre within 5
  pixels. This is a method your Project II team can use on the real data.
- **Your own masks.** Project II has no masks. Make rough ones from the
  dots: threshold the image (Otsu), then run `watershed` with one marker at
  each given centre instead of at the distance hilltops. How close do they
  get to BBBC039's hand-drawn ones (use `match_count`)? When we tried it on
  the 50 validation images, the F1 was about 0.89.
- **Tile size.** Try 128-pixel tiles every 96. Is the seam problem worse?

In [ ]:
# Your stretch work here.